<a href="https://colab.research.google.com/github/shelkekunal/Practice_of_ML/blob/main/keras_hyperparameter_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("/content/diabetes.csv")
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72.0,35,169.5,33.6,0.627,50,1
1,1,85,66.0,29,102.5,26.6,0.351,31,0
2,8,183,64.0,32,169.5,23.3,0.672,32,1
3,1,89,66.0,23,94.0,28.1,0.167,21,0
4,0,137,40.0,35,168.0,43.1,2.288,33,1


In [ ]:
df.shape

(768, 9)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    float64
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    float64
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(4), int64(5)
memory usage: 54.1 KB


In [ ]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.495990
BloodPressure,0.174469
SkinThickness,0.295138
Insulin,0.377081
BMI,0.315577
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [ ]:
x = df.drop('Outcome', axis=1)
y = df['Outcome']

In [ ]:
x_scaled.shape

(768, 8)

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y , test_size=0.2, random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)


In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input

In [ ]:
model = Sequential([
    Input(shape=(8,)),
    Dense(32, activation='relu'),
    Dense(1, activation = 'sigmoid')
])

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
history = model.fit(x_train_scaled, y_train,epochs=100, batch_size= 32, validation_split=0.2)

Epoch 1/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6273 - loss: 0.6648 - val_accuracy: 0.6260 - val_loss: 0.6448
Epoch 2/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7169 - loss: 0.5605 - val_accuracy: 0.7073 - val_loss: 0.5364
Epoch 3/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7597 - loss: 0.5012 - val_accuracy: 0.7561 - val_loss: 0.4833
Epoch 4/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7862 - loss: 0.4687 - val_accuracy: 0.7642 - val_loss: 0.4588
Epoch 5/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7902 - loss: 0.4519 - val_accuracy: 0.7724 - val_loss: 0.4411
Epoch 6/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7963 - loss: 0.4399 - val_accuracy: 0.7886 - val_loss: 0.4279
Epoch 7/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8004 - loss: 0.4309 - val_accuracy: 0.7967 - val_loss: 0.4209
Epoch 8/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8065 - loss: 0.4241 - val_accuracy: 0.8049 - 

In [ ]:
!pip install -q -U keras-tuner


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 3.7 MB/s eta 0:00:00


In [ ]:
import keras_tuner as kt

In [ ]:
def build_model(hp):
  model = Sequential()
  model.add(Input(shape=(8,))),
  model.add(Dense(32, activation='relu'))
  model.add(Dense(1, activation='sigmoid'))

  optimizer = hp.Choice('optimizer', values=['adam', 'sgd','RMSprop','adamW'])

  model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
  return model

In [ ]:
tuner= kt.RandomSearch(build_model,
                       objective='val_accuracy',
                       max_trials=5)

Reloading Tuner from ./untitled_project/tuner0.json


In [ ]:
tuner.search(x_train_scaled,y_train, epochs=5,
             validation_split=0.2,
             validation_data=(x_test,y_test))

In [ ]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'sgd'}

In [ ]:
model = tuner.get_best_models(num_models=1)[0]


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(x_train_scaled, y_train, epochs=100,initial_epoch=6 , batch_size=32, validation_split=0.2)

Epoch 7/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.6762 - loss: 0.6274 - val_accuracy: 0.6667 - val_loss: 0.6229
Epoch 8/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7026 - loss: 0.6075 - val_accuracy: 0.6911 - val_loss: 0.6046
Epoch 9/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7108 - loss: 0.5911 - val_accuracy: 0.6829 - val_loss: 0.5883
Epoch 10/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7230 - loss: 0.5765 - val_accuracy: 0.6748 - val_loss: 0.5742
Epoch 11/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7271 - loss: 0.5640 - val_accuracy: 0.6911 - val_loss: 0.5613
Epoch 12/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7434 - loss: 0.5530 - val_accuracy: 0.6992 - val_loss: 0.5506
Epoch 13/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7515 - loss: 0.5435 - val_accuracy: 0.6992 - val_loss: 0.5405
Epoch 14/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7515 - loss: 0.5348 - val_accuracy

In [ ]:
# to find optimun no. of neurons

In [ ]:
def build_model(hp):

  model=Sequential()

  units = hp.Int('units', min_value=8, max_value=128, step=8)
  model.add(Dense(units=units, activation='relu'))
  model.add(Dense(1, activation='sigmoid'))

  model.compile(optimizer='sgd', loss='binary_crossentropy', metrics=['accuracy'])

  return model

In [ ]:
tuner2 = kt.RandomSearch(build_model,
                         objective='val_accuracy',
                         max_trials=5,
                         directory = 'mydir')

In [ ]:
tuner2.search(x_train_scaled, y_train, epochs=5,
              validation_split=0.2,
              validation_data=(x_test,y_test))

Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.3571428656578064

Best val_accuracy So Far: 0.4545454680919647
Total elapsed time: 00h 00m 13s


In [ ]:
tuner2.get_best_hyperparameters()[0].values

{'units': 16}